# IRS Data Analysis
#### About:
- This notebook loads the IRS datasets, processes the amount variables, and performs a descriptive analysis.
- All steps follow the instructions set in data/Tutorial_regressions_2025 but with a python implementation.

In [1]:
# Data manipulation libraries
import pandas as pd
import os

# Data visualization libraries
import matplotlib.pyplot as plt
import seaborn as sns

## Load Data

In [2]:
# Load the dataset

file_path = 'data/IRS2020.csv'

try:
    df = pd.read_csv(file_path, sep=';')
    print("Data loaded successfully.")
    display(df.head())
except FileNotFoundError:
    print(f"Error: File not found at {file_path}")

Data loaded successfully.


,STATEFIPS,STATE,ZIPCODE,NumofReturns,TotalIncome,Salaries,NumofRetwithDivis,Divis,NumofRetwithRE,Retax
0,1,AL,35004,5420.0,331183.0000,263184.0,620.0,1250.0,280.0,330.0
1,1,AL,35005,3440.0,139266.0001,111863.0,180.0,260.0,130.0,157.0
2,1,AL,35006,1230.0,66755.0001,49973.0,100.0,132.0,20.0,20.0
3,1,AL,35007,12600.0,776780.0000,599076.0,1720.0,4974.0,730.0,1210.0
4,1,AL,35010,8280.0,467529.0000,301453.0,850.0,9786.0,360.0,943.0


## (a) Data Preprocessing

Convert all amount variables (those not starting with 'Num' and not identifiers) into dollar amounts by multiplying by 1000.

In [3]:
# Identify columns to convert
# Identifiers: STATEFIPS, STATE, ZIPCODE
# Count variables start with 'Num'

identifiers = ['STATEFIPS', 'STATE', 'ZIPCODE']
amount_cols = [col for col in df.columns if not col.startswith('Num') and col not in identifiers]

print(f"Amount variables to convert: {amount_cols}")

# Convert to dollars
for col in amount_cols:
    df[col] = df[col] * 1000

df[amount_cols].head()

Amount variables to convert: ['TotalIncome', 'Salaries', 'Divis', 'Retax']


,TotalIncome,Salaries,Divis,Retax
0,331183000.0,263184000.0,1250000.0,330000.0
1,139266000.1,111863000.0,260000.0,157000.0
2,66755000.1,49973000.0,132000.0,20000.0
3,776780000.0,599076000.0,4974000.0,1210000.0
4,467529000.0,301453000.0,9786000.0,943000.0


## Descriptive Analysis

Calculate Min, Max, Mean, Median, and Standard Deviation for all amount variables.

In [4]:
stats = df[amount_cols].agg(['min', 'max', 'mean', 'median', 'std'])

print("Descriptive Statistics (in Dollars):")
display(stats)

Descriptive Statistics (in Dollars):


,TotalIncome,Salaries,Divis,Retax
min,5.770000e+05,0.000000e+00,0.000000e+00,0.000000e+00
max,3.692462e+10,1.797653e+10,2.679518e+09,4.377930e+08
mean,4.626111e+08,2.994217e+08,1.024084e+07,3.675470e+06
median,1.249440e+08,8.274200e+07,1.119000e+06,2.910000e+05
std,8.501718e+08,5.034533e+08,4.103333e+07,9.947829e+06


## (b) Household Averages

Create variables of household (HH) averages in thousands of $ per ZIP code (Amount/Number; NumofReturn – total number of returns; for dividends and real estate (RE) taxes only calculate averages for those HH that file they have dividends or pay RE taxes) of income, salary income, dividends, and real estate taxes paid.

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

# Note: df[amount_cols] are already in dollars from step (a)
# Instructions ask for averages in thousands of $

df['HH_Income'] = (df['TotalIncome'] / 1000) / df['NumofReturns']
df['HH_Salaries'] = (df['Salaries'] / 1000) / df['NumofReturns']

# For dividends and RE taxes, only calculate for those HH that file they have dividends or pay RE taxes
df['HH_Dividends'] = (df['Divis'] / 1000) / df['NumofRetwithDivis']
df['HH_RE_Taxes'] = (df['Retax'] / 1000) / df['NumofRetwithRE']

hh_cols = ['HH_Income', 'HH_Salaries', 'HH_Dividends', 'HH_RE_Taxes']

print("Distributions of Household Averages (in thousands of $):")
display(df[hh_cols].describe())

# Boxplots of household averages across ZIP codes
plt.figure(figsize=(12, 6))
df[hh_cols].boxplot()
plt.title("Boxplots of Household Averages across ZIP codes")
plt.ylabel("Thousands of $")
plt.show()

ModuleNotFoundError: No module named 'seaborn'

## (c) Highest and Lowest Averages

Report the ZIP codes with the highest and the lowest average HH income, HH Salaries, HH Dividends, and HH Real estate taxes paid.

In [ ]:
for col in hh_cols:
    # Filter out potential Infs/NaNs from division by zero if any (though NumofReturns should be > 0)
    valid_df = df[df[col].notna() & (df[col] != float('inf'))]
    
    highest = valid_df.loc[valid_df[col].idxmax()]
    lowest = valid_df.loc[valid_df[col].idxmin()]
    
    print(f"--- {col} ---")
    print(f"Highest: ZIP {int(highest['ZIPCODE'])} ({highest['STATE']}) - ${highest[col]:.2f}k")
    print(f"Lowest:  ZIP {int(lowest['ZIPCODE'])} ({lowest['STATE']}) - ${lowest[col]:.2f}k\n")

## (d) Fraction of Households Receiving Dividends

Calculate the fraction of households receiving dividends (NumofRetwithdivis/NumofReturn). Report the distribution of this variable. Plot a scatterplot of HH income and fraction of HH receiving dividends. Also with colors by US State.

In [ ]:
df['Fraction_Divis'] = df['NumofRetwithDivis'] / df['NumofReturns']

print("Distribution of Fraction of HH receiving dividends:")
display(df['Fraction_Divis'].describe())

# Scatterplot of HH income and fraction of HH receiving dividends
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='HH_Income', y='Fraction_Divis', alpha=0.4)
plt.title("HH Income vs Fraction of HH Receiving Dividends")
plt.xlabel("HH Average Income (Thousands of $)")
plt.ylabel("Fraction of HH receiving dividends")
plt.show()

# Plot with colors by US State (Top 10 states for clarity)
top_states = df['STATE'].value_counts().nlargest(10).index
plt.figure(figsize=(12, 8))
sns.scatterplot(data=df[df['STATE'].isin(top_states)], 
                x='HH_Income', y='Fraction_Divis', hue='STATE', alpha=0.5)
plt.title("HH Income vs Fraction of HH Receiving Dividends (Top 10 States)")
plt.xlabel("HH Average Income (Thousands of $)")
plt.ylabel("Fraction of HH receiving dividends")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()